# Day 3 — RAG and Workflow Agents

**Aim:** build a small retrieval-augmented agent that answers using a local knowledge base and cites the retrieved chunks.

In [9]:
from pathlib import Path
import sys

ROOT = Path.cwd()
# If running from notebooks folder, move one level up so src/ is importable.
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from llm_router import LLMRouter, ask
router = LLMRouter()
print('Router ready. Provider order: hosted APIs -> Ollama -> mock fallback')
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

Router ready. Provider order: hosted APIs -> Ollama -> mock fallback


## 1. Load the knowledge base

In [10]:
kb_path = ROOT / 'data' / 'campus_knowledge_base.csv'
kb = pd.read_csv(kb_path)
kb

,id,title,content,source
0,1,Library opening,"The Brynmor Jones Library offers study spaces,...",Bootcamp synthetic campus KB
1,2,International student support,International students may need guidance on vi...,Bootcamp synthetic campus KB
2,3,Industrial visit preparation,"Before an industrial visit, students should co...",Bootcamp synthetic campus KB
3,4,Responsible AI practice,Responsible AI practice includes privacy prote...,Bootcamp synthetic campus KB
4,5,Agentic AI prototype,A student agent prototype should have a bounde...,Bootcamp synthetic campus KB
5,6,Research assistant use case,A research assistant agent can summarise docum...,Bootcamp synthetic campus KB
6,7,Career helper use case,A career helper agent can help students draft ...,Bootcamp synthetic campus KB
7,8,Travel planner use case,"A travel planner agent can compare routes, che...",Bootcamp synthetic campus KB


## 2. Build a simple retriever

In [11]:
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform((kb['title'] + ' ' + kb['content']).tolist())

def retrieve(query: str, k: int = 3):
    q = vectorizer.transform([query])
    scores = cosine_similarity(q, X).ravel()
    top = scores.argsort()[::-1][:k]
    results = kb.iloc[top].copy()
    results['score'] = scores[top]
    return results[['id','title','content','source','score']]

retrieve('What should students prepare before an industrial visit?')

,id,title,content,source,score
2,3,Industrial visit preparation,"Before an industrial visit, students should co...",Bootcamp synthetic campus KB,0.496521
6,7,Career helper use case,A career helper agent can help students draft ...,Bootcamp synthetic campus KB,0.152159
1,2,International student support,International students may need guidance on vi...,Bootcamp synthetic campus KB,0.044653


## 3. Generate an answer from retrieved context

In [12]:
def rag_answer(question: str) -> str:
    results = retrieve(question, k=3)
    context = '\n\n'.join(f"Source {row.id} — {row.title}: {row.content}" for row in results.itertuples())
    prompt = f'''Answer the student question using only the context below.
If the answer is not present, say what is missing.
Cite source ids in square brackets, for example [Source 3].

Context:
{context}

Question: {question}
'''
    response = router.chat([{'role':'system','content':'You are a careful RAG assistant.'}, {'role':'user','content':prompt}])
    return response.text

print(rag_answer('How should we prepare for an industrial visit?'))

MOCK LLM RESPONSE

No hosted API key or local Ollama server was detected, so this deterministic classroom fallback is being used.

I understood the request as: Answer the student question using only the context below. If the answer is not present, say what is missing. Cite source ids in square...

Suggested answer structure:
1. Clarify the user goal.
2. Break the task into small steps.
3. Use only the tools/data needed.
4. Check the result and state limitations.
5. Ask for human approval before any sensitive action.

Provider errors seen during fallback:
- ollama: 404 Client Error: Not Found for url: http://localhost:11434/api/chat


## 4. Workflow vs agent

The previous function is a workflow: retrieve → prompt → answer. Add an agentic decision: if retrieval confidence is low, ask a clarifying question.

In [13]:
def rag_agent(question: str, threshold: float = 0.12) -> str:
    results = retrieve(question, k=3)
    if float(results['score'].max()) < threshold:
        return 'I do not have enough relevant context. Please clarify the topic, organisation or task.'
    return rag_answer(question)

print(rag_agent('Can the agent book my flight?'))

MOCK LLM RESPONSE

No hosted API key or local Ollama server was detected, so this deterministic classroom fallback is being used.

I understood the request as: Answer the student question using only the context below. If the answer is not present, say what is missing. Cite source ids in square...

Suggested answer structure:
1. Clarify the user goal.
2. Break the task into small steps.
3. Use only the tools/data needed.
4. Check the result and state limitations.
5. Ask for human approval before any sensitive action.

Provider errors seen during fallback:
- ollama: 404 Client Error: Not Found for url: http://localhost:11434/api/chat


## 5. Team task

Add three rows to the knowledge base for your capstone idea. Then run five test questions and record whether the agent cited the right source.